# Session A — measurement integrity figures

Run first: `python -m archive.impact_eda.impact_session_a --through 17`

Outputs: `eda/outputs/impact_eda/figures/session_a/`

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from processing_data.paths import COURSE_RAW

ROOT = Path.cwd()
if not (ROOT / "archive" / "impact_eda" / "impact_session_a.py").exists():
    ROOT = ROOT.parents[2]
OUT = ROOT / "eda" / "outputs" / "impact_eda"
FIG = OUT / "figures" / "session_a"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "#f8f8f8"})

def save(fig, name):
    p = FIG / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved", p.relative_to(ROOT))

In [ ]:
# Fig 1 — H2 robustness forest (annual unusual → harvest)
h2 = pd.read_csv(OUT / "stage15_h2_robustness.csv")
rows = h2[h2.term.str.contains("unusual", case=False) & ~h2.term.str.contains("const")].copy()
rows["label"] = rows["spec"] + "\n" + rows["term"].str.replace("_dm", "")
fig, ax = plt.subplots(figsize=(7, 4))
y = np.arange(len(rows))
ax.barh(y, rows.coef, color="#2980b9", height=0.6)
ax.axvline(0, color="gray", lw=1)
ax.set_yticks(y, rows.spec)
ax.set_xlabel("Within-county coef (log harvested area)")
ax.set_title("SA-H1: annual unusual flood → harvest (all specs n.s. at p<0.05)")
for i, (_, r) in enumerate(rows.iterrows()):
    ax.text(r.coef, i, f"  p={r.pvalue:.3f}", va="center", fontsize=9)
save(fig, "fig01_h2_robustness_forest.png")

In [ ]:
# Fig 2 — H3 harvest vs plant coefficients by spec
h3 = pd.read_csv(OUT / "stage15_h3_robustness.csv")
harv = h3[h3.term.str.contains("harv", case=False)].copy()
plant = h3[h3.term.str.contains("plant", case=False)].copy()
specs = list(harv.spec.unique())
x = np.arange(len(specs))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, plant.set_index("spec").reindex(specs).coef, w, label="plant 4–6", color="#95a5a6")
ax.bar(x + w/2, harv.set_index("spec").reindex(specs).coef, w, label="harvest 9–11", color="#c0392b")
ax.axhline(0, color="gray")
ax.set_xticks(x, specs)
ax.set_ylabel("Within-county coef")
ax.set_title("SA-H2: seasonal windows (harvest negative; survives pre2022 — interpret with detection seasonality)")
ax.legend(loc="lower right")
save(fig, "fig02_h3_seasonal_specs.png")

In [ ]:
# Fig 3 — Lag heatmap (full sample)
lags = pd.read_csv(OUT / "stage16_monthly_lag_correlations.csv")
full = lags[lags.period == "full"].pivot(index="driver", columns="lag_months", values="spearman_r")
fig, ax = plt.subplots(figsize=(6, 3.5))
im = ax.imshow(full.values, aspect="auto", cmap="RdBu_r", vmin=-0.3, vmax=0.3)
ax.set_xticks(range(len(full.columns)), full.columns)
ax.set_xticklabels([f"lag {c}m" for c in full.columns])
ax.set_yticks(range(len(full.index)), full.index)
ax.set_title("SA-H3: county-month flood vs lagged drivers (full period)")
fig.colorbar(im, ax=ax, label="Spearman r")
save(fig, "fig03_lag_heatmap_full.png")

In [ ]:
# Fig 4 — Flood-exposed pop vs OCHA assessed (cross-section)
panel = pd.read_csv(OUT / "stage17_exposure_ocha_panel.csv")
snap = panel.sort_values("year").groupby("adm2_pcode", observed=True).tail(1)
ocha = pd.read_csv(OUT / "stage6_ocha_20251130_county.csv").rename(columns={"Admin2_PCODE": "adm2_pcode"})
admin2 = gpd.read_file(COURSE_RAW / "Administrative boundaries" / "ssd_admin2.geojson")
gdf = admin2.merge(snap[["adm2_pcode", "flood_exposed_pop", "ocha_assessed"]], on="adm2_pcode", how="left")
gdf["log_exposed"] = np.log1p(gdf["flood_exposed_pop"].fillna(0))
fig, ax = plt.subplots(figsize=(7, 8))
gdf.plot(column="log_exposed", ax=ax, legend=True, missing_kwds={"color": "#ddd"},
         legend_kwds={"label": "log1p(flood-exposed pop)", "shrink": 0.6})
unassessed = gdf[~gdf.ocha_assessed.fillna(False)]
unassessed.boundary.plot(ax=ax, color="black", linewidth=0.8, linestyle="--")
ax.set_title("SA-H4: modelled flood-exposed population; dashed = OCHA unassessed (20251130)")
ax.axis("off")
save(fig, "fig04_exposed_pop_assessment_map.png")

In [ ]:
# Fig 5 — Regime overlay: national unusual vs Victoria level (monthly)
hydro = pd.read_csv(OUT / "stage16_hydro_monthly.csv")
daily = pd.read_csv(OUT / "stage11_annual_pixel_days.csv")
u = daily[daily.flood_type_label == "unusual"]
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(u.year, u.flooded_pixel_count / 1e6, color="#3498db", alpha=0.7, label="unusual pixel-days (M)")
ax1.axvline(2021.5, color="red", linestyle="--", label="post-2020 regime")
ax1.set_ylabel("National unusual pixel-days (millions)")
ax2 = ax1.twinx()
vic = hydro.dropna(subset=["lake_victoria"])
vic_y = vic.groupby("year", observed=True)["lake_victoria"].mean()
ax2.plot(vic_y.index, vic_y.values, color="#e67e22", marker="o", label="Lake Victoria (mean monthly)")
ax2.set_ylabel("Lake level (m)")
ax1.set_title("Post-2020 mask jump vs upstream lake (context for lag / season claims)")
fig.legend(loc="upper left")
save(fig, "fig05_regime_lake_overlay.png")